In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Lenovo/Downloads/PL_team_matches.csv")
df["Date"] = pd.to_datetime(df["Date"])
df_home = df[df["HomeAway"] == "Home"].copy()              # Filtered to predict if only the home team will score 2 or more goals
df_home["Target"] = (df_home["FTHG"] >= 2).astype(int)

df.shape
df.head()

,MatchID,Season,Date,Time,Team,Opponent,HomeAway,Goals,GoalsConceded,HTGoals,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,1,2021/2022,2021-08-13,20:00,Brentford,Arsenal,Home,2,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2021/2022,2021-08-13,20:00,Arsenal,Brentford,Away,0,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2021/2022,2021-08-14,12:30,Man United,Leeds,Home,5,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,2021/2022,2021-08-14,12:30,Leeds,Man United,Away,1,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2021/2022,2021-08-14,15:00,Burnley,Brighton,Home,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
#The split is sone by date to ensure no leakage
train_end = "2024-08-01"
val_end = "2025-08-01"

train = df_home[df_home["Date"] < train_end]

validation = df_home[(df_home["Date"] >= train_end) & 
                     (df_home["Date"] < val_end)]

test = df_home[df_home["Date"] >= val_end]

print(train.shape, validation.shape, test.shape)

(1140, 184) (380, 184) (380, 184)


In [3]:
print(len(set(train["MatchID"]) & set(validation["MatchID"])))
print(len(set(validation["MatchID"]) & set(test["MatchID"])))
print(len(set(train["MatchID"]) & set(test["MatchID"])))         # If all are equal to zero then theres no  overlap and hence no leakage

0
0
0


In [4]:
print(len(train) + len(validation) + len(test) == len(df_home))

True


In [11]:
total_matches = len(df_home)
print(f"Train percentage:      {len(train) / total_matches:.1%}")
print(f"Validation percentage: {len(validation) / total_matches:.1%}")
print(f"Test percentage:       {len(test) / total_matches:.1%}")

Train percentage:      60.0%
Validation percentage: 20.0%
Test percentage:       20.0%


In [14]:
# FTHG, FTAG and FTR are the result so I need to make sure to exclude them from leaking into X
# Define X as what goes into the model and Y as what it predicts

feature_cols = ['HS', 'AS', 'HST', 'AST']

X_train = train[feature_cols]
Y_train = train["Target"]

X_val = validation[feature_cols]
Y_val = validation["Target"]

X_test = test[feature_cols]
Y_test = test["Target"]

print("The model is told how many shots and shots on target were recorded and then is told after then the team scored 2 or more goals")

The model is told how many shots and shots on target were recorded and then is told after then the team scored 2 or more goals
